# 01 EDA: разведочный анализ логов Ozon Similar Products

Цель: понять, готовы ли текущие логи к preprocessing, sessionization и первому co-visitation baseline.

Ноутбук оформлен как отчёт: расчётные ячейки подготавливают данные без вывода, а каждая демонстрационная ячейка выводит одну таблицу.


## 1. Импорты, путь проекта и конфиги

Сначала настраиваем окружение: определяем корень проекта, добавляем `src` в путь импорта Python, подключаем библиотеки и находим директории с parquet-данными.


In [ ]:
import sys
from pathlib import Path
from datetime import date

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import polars as pl

from ozon_similar_products.data import load_products
from ozon_similar_products.config import get_path_from_config, load_configs
from ozon_similar_products.data.readers import find_parquet_payload_dir
from ozon_similar_products.features.item_popularity import ItemPopularityBuilder
from ozon_similar_products.diagnostics.profiling import (
    null_profile,
    parquet_dataset_overview,
    parquet_partition_profile,
    partition_row_counts,
    schema_overview,
)
from ozon_similar_products.data.schemas import KNOWN_ACTION_TYPES
from ozon_similar_products.diagnostics.session_checks import (
    add_session_markers,
    time_diff_summary,
)


In [ ]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)


загружаем проектные настройки и находим фактические папки с товарами и пользовательскими действиями.


In [ ]:
PROJECT_CONFIG = load_configs(project_root=PROJECT_ROOT)
DATA_CONFIG = PROJECT_CONFIG["data"]

PRODUCTS_BASE_DIR = get_path_from_config(
    PROJECT_CONFIG,
    section="data",
    key="product_information_dir",
)
USER_ACTIONS_BASE_DIR = get_path_from_config(
    PROJECT_CONFIG,
    section="data",
    key="user_actions_dir",
)

PRODUCTS_ROOT = find_parquet_payload_dir(
    base_dir=PRODUCTS_BASE_DIR,
    payload_root_names=DATA_CONFIG["product_information"]["payload_root_names"],
    parquet_glob=DATA_CONFIG["product_information"]["parquet_glob"],
)
ACTIONS_ROOT = find_parquet_payload_dir(
    base_dir=USER_ACTIONS_BASE_DIR,
    payload_root_names=DATA_CONFIG["user_actions"]["payload_root_names"],
    parquet_glob=DATA_CONFIG["user_actions"]["parquet_glob"],
)

paths_check = pl.DataFrame(
    {
        "объект": [
            "project_root",
            "src_path",
            "products_base_dir",
            "user_actions_base_dir",
            "products_root",
            "actions_root",
        ],
        "путь": [
            str(PROJECT_ROOT),
            str(SRC_PATH),
            str(PRODUCTS_BASE_DIR),
            str(USER_ACTIONS_BASE_DIR),
            str(PRODUCTS_ROOT),
            str(ACTIONS_ROOT),
        ],
    }
)


In [ ]:
paths_check


## 2. Обзор справочника товаров


загружаем `product_information` и рассчитываем три диагностические таблицы: общую сводку, схему и пропуски.


In [ ]:
products = load_products(config=PROJECT_CONFIG)

product_schema = schema_overview(products)
product_nulls = null_profile(products)
product_summary = pl.DataFrame(
    {
        "метрика": [
            "строк",
            "уникальных item_id",
            "строк с дублями item_id",
            "уникальных category_id",
            "уникальных category_name",
            "уникальных brand",
            "уникальных type",
        ],
        "значение": [
            products.height,
            products.select(pl.col("item_id").n_unique()).item(),
            products.height - products.select(pl.col("item_id").n_unique()).item(),
            products.select(pl.col("category_id").n_unique()).item(),
            products.select(pl.col("category_name").n_unique()).item(),
            products.select(pl.col("brand").n_unique()).item(),
            products.select(pl.col("type").n_unique()).item(),
        ],
    }
)


Общая сводка показывает размер справочника, уникальность ключей и разнообразие товарных атрибутов.


In [ ]:
product_summary


смотрим фактические колонки и типы данных в справочнике товаров.


In [ ]:
product_schema


смотрим в каких колонках есть отсутствующие значения и насколько они значимы по доле.


In [ ]:
product_nulls


 `item_id` уникален, дублей по ключу нет. Технические пропуски есть только в неключевых полях `brand` и `category_name`, поэтому сам каталог технически пригоден для join-ов, но его согласованность с логами проверяется отдельно.


In [ ]:
top_categories = (
    products.group_by(["category_id", "category_name"])
    .agg(pl.len().alias("items"))
    .sort("items", descending=True)
    .head(30)
)

top_brands = (
    products.group_by("brand")
    .agg(pl.len().alias("items"))
    .sort("items", descending=True)
    .head(30)
)

top_types = (
    products.group_by("type")
    .agg(pl.len().alias("items"))
    .sort("items", descending=True)
    .head(30)
)


выведем топ категорий

In [ ]:
top_categories


топ брендов

In [ ]:
top_brands

топ типов товаров


In [ ]:
top_types


Каталог неоднородный: крупные группы включают зоотовары, бытовую химию, детские товары, электронику и fresh-категории. Для fallback и правил качества лучше использовать стабильные идентификаторы (`category_id`, `type`) и аккуратно обработанный бренд, а текстовые названия категорий оставлять для интерпретации.

В топе брендов видно, что `Нет бренда` является одной из крупных групп. Это не полноценный бренд, а служебное значение отсутствующего бренда, поэтому для анализа рекомендаций, fallback-правил и business rules его нужно объединять с `null` в категорию неизвестного бренда.


## 3. Логи действий: parquet metadata и дневные партиции

Логи пользовательских действий слишком большие для полной загрузки в память, поэтому сначала смотрим только parquet metadata: количество файлов, строк, размер данных, даты и распределение по `action_type`.


читаем parquet metadata и собирает агрегаты по датам и типам действий без полной материализации логов.


In [ ]:
actions_overview = parquet_dataset_overview(ACTIONS_ROOT)
actions_file_profile = parquet_partition_profile(ACTIONS_ROOT)
date_action_counts = partition_row_counts(ACTIONS_ROOT)

date_summary = (
    date_action_counts.group_by("date")
    .agg(
        pl.col("rows").sum().alias("rows"),
        pl.col("files").sum().alias("files"),
        pl.col("file_size_bytes").sum().alias("file_size_bytes"),
    )
    .sort("date")
)

date_summary_start = date_summary.head()
date_summary_end = date_summary.tail()

action_rows_from_metadata = (
    date_action_counts.group_by("action_type")
    .agg(pl.col("rows").sum().alias("rows"))
    .with_columns((pl.col("rows") / pl.col("rows").sum()).alias("share"))
    .sort("rows", descending=True)
)


Общий обзор логов показывает число parquet-файлов, строк и суммарный размер датасета.


In [ ]:
actions_overview


Первые даты периода нужны, чтобы убедиться, с какого дня начинается наблюдение.


In [ ]:
date_summary_start


Последние даты периода нужны, чтобы проверить когда заканчиваются наблюдения.


In [ ]:
date_summary_end


Лог содержит 561,836,992 строки в 305 parquet-файлах за период 2024-03-01 — 2024-04-30.


делаем короткую проверку состояния логов по дням: период, количество файлов, строк и наличие известных типов событий. Health-check показывает, достаточно ли корректно разложены дневные партиции для дальнейших lazy-проверок.



In [ ]:
partition_health = pl.DataFrame(
    {
        "метрика": [
            "минимальная дата",
            "максимальная дата",
            "дней",
            "parquet-файлов",
            "строк",
            "минимум строк за день",
            "максимум строк за день",
            "найденные известные action_type",
        ],
        "значение": [
            str(date_summary.select(pl.col("date").min()).item()),
            str(date_summary.select(pl.col("date").max()).item()),
            str(date_summary.height),
            str(actions_overview["files"].item()),
            str(actions_overview["rows"].item()),
            str(date_summary.select(pl.col("rows").min()).item()),
            str(date_summary.select(pl.col("rows").max()).item()),
            ", ".join(sorted(set(action_rows_from_metadata["action_type"].to_list()) & set(KNOWN_ACTION_TYPES))),
        ],
    }
)


In [ ]:
partition_health


Дневные партиции покрывают весь наблюдаемый период, а известные `action_type` присутствуют в данных. Это достаточная основа для дальнейших lazy-агрегаций и дневных проверок без первичного признака пропавших партиций.


## 4. Lazy scan и схема логов

Для дальнейших проверок создаём `LazyFrame` поверх всех parquet-файлов. Это позволяет выполнять агрегации по нужным колонкам без материализации полного датасета.


создаём ленивый scan всех parquet-файлов и готовим таблицу с фактической схемой логов.


In [ ]:
user_actions_lf = pl.scan_parquet(str(ACTIONS_ROOT / "**" / "*.parquet"), hive_partitioning=True)
actions_schema = schema_overview(user_actions_lf)


Схема логов показывает, какие поля доступны для EDA, preprocessing и sessionization.


In [ ]:
actions_schema


Схема логов содержит все поля, необходимые для EDA и будущего preprocessing: `user_id`, `timestamp`, `action_type`, `widget_name`, `search_query`, `item_id` и `date`. Типы данных выглядят согласованно с назначением колонок.


## 5. Анализ типов событий, пропусков и widget-контекста

На этом этапе отделяем прямые товарные действия от поисковых событий и смотрим, как распределены `action_type`, `item_id`, `search_query` и `widget_name`. Первый co-visitation baseline строится на взаимодействиях с конкретным товаром, поэтому `search` анализируется отдельно.


In [ ]:
key_action_columns = ["user_id", "timestamp", "action_type", "widget_name", "search_query", "item_id"]
actions_key_lf = user_actions_lf.select(key_action_columns)

direct_action_types = ["view", "click", "favorite", "to_cart"]

action_scope = pl.DataFrame(
    {
        "action_type": ["view", "click", "favorite", "to_cart", "search"],
        "используется_в_direct_item_graph": [True, True, True, True, False],
        "роль": [
            "просмотр товара",
            "клик по товару",
            "добавление в избранное",
            "добавление в корзину",
            "поисковый интент без обязательного item_id",
        ],
    }
)


зафиксируем, какие типы событий входят в первый товарный граф, а какие остаются отдельным источником сигналов.


In [ ]:
action_scope


Для дальнейшего анализа выделены только необходимые колонки. Direct item graph будет строиться по `view`, `click`, `favorite`, `to_cart`; `search` исключается из первого item-to-item графа, потому что не содержит конкретного `item_id`.


In [ ]:
actions_by_type = action_rows_from_metadata.select(["action_type", "rows", "share"])


Распределение типов событий показывает объём каждого сигнала и помогает увидеть перекос в сторону `view`.


In [ ]:
actions_by_type


Все ожидаемые типы событий присутствуют. Доля `view` максимальна, поэтому raw co-visitation без нормализации и quality gates будет сильно смещён в сторону популярных просмотров.


In [ ]:
critical_action_nulls = null_profile(
    actions_key_lf,
    columns=[
        "user_id",
        "timestamp",
        "action_type",
        "item_id",
        "search_query",
        "widget_name",
    ],
)


посмотрим пропуски в критичных колонках пользовательских действий


In [ ]:
critical_action_nulls


In [ ]:
widget_action_profile = (
    actions_key_lf
    .select(["action_type", "widget_name"])
    .group_by(["action_type", "widget_name"])
    .agg(pl.len().alias("rows"))
    .with_columns((pl.col("rows") / pl.col("rows").sum()).alias("share"))
    .sort(["action_type", "rows"], descending=[False, True])
    .collect()
)


посмотрим из каких интерфейсных контекстов приходят разные типы действий.


In [ ]:
widget_action_profile


`widget_name` показывает, из какого интерфейсного контекста пришло событие. Это важно для диагностики: одинаковый `action_type` из разных мест интерфейса может иметь разную силу пользовательского намерения.


Эта ячейка отдельно смотрит widget-контекст только для direct item events с непустым `item_id`.


In [ ]:
direct_widget_profile = (
    actions_key_lf
    .filter(pl.col("action_type").is_in(direct_action_types))
    .filter(pl.col("item_id").is_not_null())
    .select(["widget_name"])
    .group_by("widget_name")
    .agg(pl.len().alias("rows"))
    .with_columns((pl.col("rows") / pl.col("rows").sum()).alias("share"))
    .sort("rows", descending=True)
    .collect()
)


видим какие виджеты дают основной поток прямых товарных взаимодействий.


In [ ]:
direct_widget_profile


Большинство товарных действий приходит из `search_catalog_listing`, а не только из карточки товара. Для MVP это не блокер, но в будущей evaluation стоит сравнить качество рекомендаций по widget-specific разрезам или хотя бы мониторить этот контекст.


зафиксируем правила обработки пропусков и спорных случаев перед preprocessing.


In [ ]:
missing_value_decisions = pl.DataFrame(
    {
        "условие": [
            "action_type = search and item_id is null",
            "action_type in view/click/favorite/to_cart and item_id is null",
            "user_id is null",
            "timestamp is null",
            "action_type is null",
            "search_query is null for non-search actions",
            "widget_name is null",
            "brand is null in product_information",
            "brand = Нет бренда in product_information",
            "category_name is null in product_information",
        ],
        "решение": [
            "допустимо; не включать search-события в direct item graph",
            "не включать в direct item events",
            "не включать в clean action events",
            "не включать в clean action events",
            "не включать в clean action events",
            "допустимо",
            "оставить как unknown widget и контролировать долю",
            "привести к единому значению missing_brand",
            "привести к единому значению missing_brand",
            "оставить category_id как основной ключ категории",
        ],
        "причина": [
            "поиск выражает query-intent, а не взаимодействие с конкретным item_id",
            "для item-to-item пары нужен конкретный товар",
            "без пользователя нельзя строить сессии и item-to-item пары",
            "без времени нельзя упорядочить события внутри сессии",
            "тип события определяет, в какую ветку обработки попадёт строка",
            "поисковый текст нужен только для search-ветки",
            "widget context полезен, но не является ключом графа",
            "null и служебное значение Нет бренда описывают один и тот же случай отсутствующего бренда",
            "null и служебное значение Нет бренда описывают один и тот же случай отсутствующего бренда",
            "название категории нужно для интерпретации, но не как ключ",
        ],
    }
)

Таблица ниже переводит наблюдения о пропусках в конкретные preprocessing-решения.


In [ ]:
missing_value_decisions


## 6. Анализ пользователей

Дальше оцениваем активность пользователей по direct item events: сколько событий и уникальных товаров приходится на пользователя, как выглядит хвост распределения и насколько велик вклад p99+ сегмента.


In [ ]:
direct_item_events_lf = (
    user_actions_lf
    .filter(pl.col("action_type").is_in(direct_action_types))
    .filter(pl.col("item_id").is_not_null())
    .select(["user_id", "date", "timestamp", "action_type", "item_id", "widget_name"])
)

user_activity_lf = direct_item_events_lf.group_by("user_id").agg(
    pl.len().alias("events"),
    pl.col("item_id").n_unique().alias("unique_items"),
    pl.col("date").n_unique().alias("active_days"),
    pl.col("timestamp").min().alias("first_event_at"),
    pl.col("timestamp").max().alias("last_event_at"),
)

user_activity_summary = user_activity_lf.select(
    pl.len().alias("users"),
    pl.col("events").sum().alias("events"),
    pl.col("events").mean().alias("events_mean"),
    pl.col("events").quantile(0.5).alias("events_p50"),
    pl.col("events").quantile(0.9).alias("events_p90"),
    pl.col("events").quantile(0.95).alias("events_p95"),
    pl.col("events").quantile(0.99).alias("events_p99"),
    pl.col("events").max().alias("events_max"),
).collect()

p99_events_threshold = user_activity_summary["events_p99"].item()
total_direct_events = user_activity_summary["events"].item()

high_activity_user_summary = (
    user_activity_lf
    .filter(pl.col("events") >= p99_events_threshold)
    .select(
        pl.lit(p99_events_threshold).alias("events_p99_threshold"),
        pl.len().alias("p99_plus_users"),
        pl.col("events").sum().alias("p99_plus_events"),
        (pl.col("events").sum() / pl.lit(total_direct_events)).alias("p99_plus_event_share"),
        pl.col("events").max().alias("max_events"),
        pl.col("unique_items").max().alias("max_unique_items"),
        (pl.col("events").max() / pl.lit(total_direct_events)).alias("top_user_event_share"),
    )
    .collect()
)

top_active_users = user_activity_lf.sort("events", descending=True).limit(50).collect()


Сводка пользовательской активности показывает распределение числа событий на пользователя.


In [ ]:
user_activity_summary


посмотрим вклад p99+ пользователей и риск перекоса item-pair counts.


In [ ]:
high_activity_user_summary


Топ активных пользователей помогает вручную проверить крайние случаи активности.


In [ ]:
top_active_users


Распределение пользовательской активности сильно скошено. меньше процента пользователей дают существенную долю direct item events, а самый активный пользователь может заметно исказить item-pair counts. Для preprocessing нужны p99+ flag/cap и контроль длины сессии.


## 7. Анализ товаров, catalog reconciliation и popularity bias

В этом блоке считаем невзвешенную популярность товаров на небольшом семпле дней и фиксируем риски popularity bias, а также сравниваем множество `item_id` из поведения со справочником


Считаем популярность на семпле из 5 дней из разных частей периода.


In [ ]:
ITEM_ACTION_TYPES = ["view", "click", "favorite", "to_cart"]
POPULARITY_SAMPLE_DATES = [
    date(2024, 3, 1),
    date(2024, 3, 15),
    date(2024, 3, 29),
    date(2024, 4, 12),
    date(2024, 4, 30),
]
POPULARITY_SAMPLE_DATES_STR = [sample_date.isoformat() for sample_date in POPULARITY_SAMPLE_DATES]

popularity_events = (
    user_actions_lf
    .select(["user_id", "date", "timestamp", "action_type", "item_id"])
    .with_columns(pl.col("date").cast(pl.String))
    .filter(pl.col("date").is_in(POPULARITY_SAMPLE_DATES_STR))
)

item_popularity_builder = ItemPopularityBuilder(item_action_types=ITEM_ACTION_TYPES)
item_popularity_all = (
    item_popularity_builder
    .build_item_popularity(popularity_events)
    .rename(
        {
            "events_count": "events",
            "views_count": "views",
            "clicks_count": "clicks",
            "favorites_count": "favorites",
            "to_cart_count": "to_carts",
        }
    )
    .with_columns(
        pl.when(pl.col("unique_users") > 0)
        .then(pl.col("events") / pl.col("unique_users"))
        .otherwise(0.0)
        .alias("events_per_user")
    )
    .sort(["events", "item_id"], descending=[True, False])
)
item_popularity_top = item_popularity_all.head(100)

total_events = int(item_popularity_all["events"].sum()) if item_popularity_all.height else 0
concentration_points = [1, 5, 10, 25, 50, 100]
popularity_concentration_table = pl.DataFrame(
    {
        "top_n": concentration_points,
        "events_share": [
            (
                float(item_popularity_all.head(top_n)["events"].sum()) / total_events
                if total_events > 0
                else 0.0
            )
            for top_n in concentration_points
        ],
    }
)

action_type_popularity = (
    popularity_events
    .filter(pl.col("item_id").is_not_null())
    .filter(pl.col("action_type").is_in(ITEM_ACTION_TYPES))
    .group_by(["action_type", "item_id"])
    .agg(
        [
            pl.len().alias("events"),
            pl.col("user_id").drop_nulls().n_unique().alias("unique_users"),
        ]
    )
    .with_columns(
        pl.when(pl.col("unique_users") > 0)
        .then(pl.col("events") / pl.col("unique_users"))
        .otherwise(0.0)
        .alias("events_per_user")
    )
    .sort(["action_type", "events", "item_id"], descending=[False, True, False])
    .group_by("action_type", maintain_order=True)
    .head(100)
)


Топ популярных товаров на семпле нужен для диагностики popularity bias. Смотрим не только `events`, но и `unique_users`, `events_per_user` и разрезы по типам действий.


In [ ]:
item_popularity_top


Концентрация показывает, какая доля событий на sample days приходится на top-k популярных товаров.


In [ ]:
popularity_concentration_table


Разрез по типам действий нужен, чтобы не смешивать `view`, `click`, `favorite` и `to_cart` в один непрозрачный счётчик.


In [ ]:
action_type_popularity


сравним множество товаров из поведения с множеством товаров из справочника


In [ ]:
behavior_item_ids = (
    user_actions_lf
    .filter(pl.col("item_id").is_not_null())
    .filter(pl.col("action_type").is_in(ITEM_ACTION_TYPES))
    .select("item_id")
    .unique()
    .collect()
)
catalog_item_ids = products.select("item_id").unique()

matched_items = behavior_item_ids.join(catalog_item_ids, on="item_id", how="inner").height
behavior_items_without_product = behavior_item_ids.join(
    catalog_item_ids,
    on="item_id",
    how="anti",
).height
catalog_items_without_behavior = catalog_item_ids.join(
    behavior_item_ids,
    on="item_id",
    how="anti",
).height

catalog_items = catalog_item_ids.height
behavior_items = behavior_item_ids.height

catalog_reconciliation = pl.DataFrame(
    {
        "метрика": [
            "catalog_items",
            "behavior_items",
            "matched_items",
            "behavior_items_without_product",
            "catalog_items_without_behavior",
            "catalog_behavior_coverage",
            "behavior_catalog_match_rate",
        ],
        "значение": [
            float(catalog_items),
            float(behavior_items),
            float(matched_items),
            float(behavior_items_without_product),
            float(catalog_items_without_behavior),
            None if catalog_items == 0 else matched_items / catalog_items,
            None if behavior_items == 0 else matched_items / behavior_items,
        ],
    },
    schema_overrides={"значение": pl.Float64},
)


Catalog reconciliation показывает пересечение и расхождения между каталогом и поведением.


In [ ]:
catalog_reconciliation


Найдена существенная несогласованность между каталогом и поведением: часть `item_id` из логов отсутствует в `product_information`. Поэтому catalog reconciliation — не просто диагностическая метрика, а блокер для финальной baseline/evaluation до синхронизации данных или фильтрации до `matched_items`.


зафиксируем решения, которые нужны, чтобы baseline не свёлся к рекомендации самых популярных товаров, а также решение по несовпадению товаров в логах и справочнике.


In [ ]:
popularity_decisions = pl.DataFrame(
    {
        "вопрос": [
            "Использовать raw pair_count как финальный score?",
            "Добавлять min_pair_count?",
            "Добавлять min_unique_users?",
            "Нормализовать популярность уже в baseline?",
            "Что делать с p99+ пользователями?",
            "Можно ли оценивать качество без catalog reconciliation?",
        ],
        "решение": [
            "нет",
            "да",
            "да",
            "да, хотя бы простым popularity normalization",
            "добавить cap или отдельный флаг перед генерацией пар",
            "нет, финальная evaluation требует согласованного item universe",
        ],
        "причина": [
            "raw pair_count усиливает самые популярные товары",
            "редкие пары дают шум",
            "один пользователь не должен создавать сильную пару",
            "иначе score будет почти popularity score",
            "экстремальная активность искажает co-visitation",
            "нельзя честно оценивать рекомендации по товарам, отсутствующим в справочнике",
        ],
    }
)


Таблица ниже переводит риски popularity bias в конкретные правила baseline/preprocessing.


In [ ]:
popularity_decisions


Для первого baseline нужны ограничения против popularity bias: `min_pair_count`, `min_unique_users`, нормализация score и обработка p99+ пользователей. Без этих решений baseline будет больше отражать популярность и технический трафик, чем реальную похожесть товаров.


## 8. Анализ поисковых запросов

Поисковые события анализируются отдельно: они не входят в первый direct item graph, но могут быть полезны для будущих co-search или query-intent признаков.


In [ ]:
search_lf = user_actions_lf.filter(pl.col("action_type") == "search").with_columns(
    pl.col("search_query").str.strip_chars().alias("search_query_trimmed")
)

search_summary = search_lf.select(
    pl.len().alias("search_events"),
    pl.col("user_id").n_unique().alias("search_users"),
    pl.when(
        pl.col("search_query_trimmed").is_not_null()
        & (pl.col("search_query_trimmed") != "")
    )
    .then(pl.col("search_query_trimmed").str.to_lowercase())
    .otherwise(None)
    .drop_nulls()
    .n_unique()
    .alias("unique_queries_normalized"),
    pl.col("search_query").null_count().alias("null_query_rows"),
    (pl.col("search_query") == "").sum().alias("empty_string_query_rows"),
    (pl.col("search_query_trimmed") == "").sum().alias("blank_query_rows"),
).collect().with_columns(
    (pl.col("null_query_rows") + pl.col("blank_query_rows")).alias("invalid_query_rows"),
    (
        (pl.col("null_query_rows") + pl.col("blank_query_rows"))
        / pl.col("search_events")
    ).alias("invalid_query_share"),
).select(
    [
        "search_events",
        "search_users",
        "unique_queries_normalized",
        "null_query_rows",
        "empty_string_query_rows",
        "blank_query_rows",
        "invalid_query_rows",
        "invalid_query_share",
    ]
)


Сводка поисковых событий показывает объём поиска и долю пустых или невалидных запросов.


In [ ]:
search_summary


Эта ячейка оставляет только валидные поисковые запросы и считает самые частые нормализованные запросы.


In [ ]:
valid_search_lf = (
    search_lf
    .filter(pl.col("search_query_trimmed").is_not_null())
    .filter(pl.col("search_query_trimmed") != "")
    .with_columns(
        pl.col("search_query_trimmed").str.to_lowercase().alias("search_query_normalized")
    )
)

top_queries = (
    valid_search_lf
    .group_by("search_query_normalized")
    .agg(pl.len().alias("rows"), pl.col("user_id").n_unique().alias("users"))
    .sort("rows", descending=True)
    .limit(50)
    .collect()
)


Топ поисковых запросов показывает основные пользовательские интенты.


In [ ]:
top_queries


Поисковые запросы в целом пригодны для будущего анализа: невалидных запросов мало. В текущем baseline `search` остаётся вне direct item graph, но его стоит сохранить как отдельный источник query-intent/co-search сигналов.


## 9. Timestamp и возможность собрать сессии в пределах дневных партиций

В этой части выполняется дневная проверка session_feasibility по всем партициям. Она проверяет, что при выбранном timeout сессии технически считаются внутри дневных партиций.

задаём session timeout и список дат для дневной проверки. В качестве стартового значения для первого co-visitation baseline будем брать тайм-аут 30 минут. Это значение не считается финально оптимальным: после построения baseline тайм-аут нужно подобрать по validation-метрике.


In [ ]:
session_timeout_minutes = 30
date_values = date_summary["date"].to_list()

проходим по дневным партициям и считаем распределение временных разрывов между событиями пользователя.


In [ ]:
session_summaries = []

for date_value in date_values:
    day_path = ACTIONS_ROOT / f"date={date_value}"
    day_lf = (
        pl.scan_parquet(str(day_path / "**" / "*.parquet"), hive_partitioning=True)
        .filter(pl.col("action_type").is_in(direct_action_types))
        .filter(pl.col("item_id").is_not_null())
        .select(["user_id", "timestamp", "item_id"])
    )
    day_summary = time_diff_summary(
        day_lf,
        timeout_minutes=session_timeout_minutes,
        group_cols="user_id",
    ).with_columns(pl.lit(date_value).alias("date"))
    session_summaries.append(day_summary)

session_feasibility = pl.concat(session_summaries).select(
    [
        "date",
        "events",
        "time_diffs",
        "sessions",
        "gaps_over_timeout",
        "p50_seconds",
        "p75_seconds",
        "p90_seconds",
        "p95_seconds",
        "p99_seconds",
    ]
)


Таблица показывает, насколько данные пригодны для sessionization по time gaps внутри дневных партиций.


In [ ]:
session_feasibility


Timestamp пригоден для расчёта time gaps внутри дневных партиций. При этом проверка day-bounded: она не доказывает корректность сессий через полночь. Production sessionization должна сортировать события пользователя по всему периоду или переносить состояние сессии между датами.


После `session_feasibility` оцениваем стартовый `max_items_per_session` по сэмплу пользователей. p99+ пользователей исключаем из этой диагностики, потому что cap хотим выбирать по обычному поведению. Для выбранных `user_id` берём все direct item events за весь период, поэтому сохраняется полный пользовательский timeline и переходы через границы дней.


In [ ]:
SESSION_LENGTH_SAMPLE_BUCKETS = 1_000
SESSION_LENGTH_SAMPLE_PER_MILLE = 50
SESSION_LENGTH_SAMPLE_SEED = 42
SESSION_LENGTH_TIMEOUT_MINUTES = session_timeout_minutes

sampled_session_users_lf = (
    user_activity_lf
    .filter(pl.col("events") < p99_events_threshold)
    .with_columns(
        (
            pl.col("user_id")
            .hash(seed=SESSION_LENGTH_SAMPLE_SEED)
            .mod(SESSION_LENGTH_SAMPLE_BUCKETS)
        ).alias("sample_bucket")
    )
    .filter(pl.col("sample_bucket") < SESSION_LENGTH_SAMPLE_PER_MILLE)
    .select("user_id")
)

sampled_session_users = sampled_session_users_lf.collect()
sampled_session_events = (
    direct_item_events_lf
    .join(sampled_session_users.lazy(), on="user_id", how="inner")
    .select(["user_id", "timestamp", "item_id"])
    .collect()
)

session_length_sample_overview = pl.DataFrame(
    {
        "sample_users": [sampled_session_users.height],
        "sample_events": [sampled_session_events.height],
        "sample_unique_items": [
            sampled_session_events.select(pl.col("item_id").n_unique()).item()
        ],
        "sample_first_event_at": [sampled_session_events.select(pl.col("timestamp").min()).item()],
        "sample_last_event_at": [sampled_session_events.select(pl.col("timestamp").max()).item()],
        "excluded_user_activity_threshold": [p99_events_threshold],
    }
)


Проверяем размер сэмпла: сколько пользователей и событий попало после исключения p99+ пользователей.


In [ ]:
session_length_sample_overview


На этом сэмпле считаем квантили количества уникальных товаров в сессии по всем direct item actions, включая `view`. Для первичного cap достаточно увидеть обычную длину сессий и хвост: `p50/p75/p90/p95/p99/max`.


In [ ]:
def summarize_session_length_quantiles_for_current_timeout(
    events: pl.DataFrame,
) -> pl.DataFrame:
    timeout_minutes = SESSION_LENGTH_TIMEOUT_MINUTES
    session_lengths = (
        add_session_markers(
            events,
            timeout_minutes=timeout_minutes,
            group_cols="user_id",
        )
        .group_by(["user_id", "session_index"])
        .agg(
            pl.len().alias("events"),
            pl.col("item_id").n_unique().alias("unique_items"),
        )
        .with_columns(pl.lit(timeout_minutes).alias("timeout_minutes"))
        .collect()
    )

    return session_lengths.select(
        pl.first("timeout_minutes"),
        pl.len().alias("sessions"),
        pl.col("events").mean().alias("events_mean"),
        pl.col("unique_items").mean().alias("unique_items_mean"),
        pl.col("unique_items").quantile(0.5).alias("unique_items_p50"),
        pl.col("unique_items").quantile(0.75).alias("unique_items_p75"),
        pl.col("unique_items").quantile(0.9).alias("unique_items_p90"),
        pl.col("unique_items").quantile(0.95).alias("unique_items_p95"),
        pl.col("unique_items").quantile(0.99).alias("unique_items_p99"),
        pl.col("unique_items").max().alias("unique_items_max"),
    )


session_length_quantiles = summarize_session_length_quantiles_for_current_timeout(
    sampled_session_events,
)


Таблица ниже показывает квантили длины сессии для timeout `30 минут` на сэмпле пользователей без p99+ активности.

In [ ]:
session_length_quantiles


Можно взять максимальную длину сессии 140. Этот параметр также можно будет подбирать после baseline.

Эта ячейка фиксирует стартовые правила sessionization перед генерацией co-visitation pairs.


In [ ]:
session_decisions = pl.DataFrame(
    {
        "вопрос": [
            "Можно ли использовать timestamp для user timelines?",
            "Начальный session timeout",
            "Max items per session",
            "Как обрабатывать very long sessions",
            "Как обрабатывать экстремально активных пользователей",
            "Нужно ли учитывать переход через полночь?",
        ],
        "решение": [
            "да",
            "30 минут",
            "оценить по user-sampled session length p95/p99",
            "обрезать или исключать по правилу, выбранному до обучения",
            "cap/flag на основе p99+ анализа",
            "да, production sessionization должна работать на полном timeline пользователя",
        ],
        "причина": [
            "timestamp есть и позволяет считать time gaps",
            "берем за стартовое не обязательно оптимальное значение",
            "cap нужен против длинного хвоста и квадратичного роста item pairs",
            "длинные сессии часто похожи на шум или ботовое поведение",
            "активные пользователи могут доминировать в pair counts",
            "текущая EDA-проверка day-bounded и не видит междневные сессии",
        ],
    }
)


Таблица ниже переводит проверку timestamp в конкретные preprocessing-решения.


In [ ]:
session_decisions


Сессионные правила сформулированы как стартовые preprocessing decisions: timeout 30 минут остаётся provisional baseline default, а `max_items_per_session` оценивается по user-sampled распределению длины сессий. Эти решения нужны до генерации co-visitation пар и затем должны быть проверены на temporal validation.


## 10. Чеклист готовности baseline и выводы EDA

В финальном блоке результаты EDA сведены в readiness checklist. Он отделяет уже закрытые проверки от ограничений, которые нужно учесть перед preprocessing, первым co-visitation baseline и особенно перед финальной evaluation.


Эта ячейка собирает итоговый чеклист готовности данных и baseline-логики.


In [ ]:
baseline_readiness = pl.DataFrame(
    {
        "проверка": [
            "данные загружены",
            "схема совпадает с контрактом",
            "timestamp корректно парсится",
            "период данных понятен",
            "дневные партиции доступны",
            "типы событий понятны",
            "типы событий для графа выбраны",
            "widget context проверен",
            "правило обработки search определено",
            "пропуски классифицированы",
            "аномалии пользователей проанализированы",
            "popularity bias проанализирован",
            "catalog-behavior reconciliation",
            "sessionization возможна",
            "можно переходить к preprocessing",
            "можно переходить к co-visitation baseline",
        ],
        "статус": [
            "готово",
            "готово",
            "готово",
            "готово",
            "готово",
            "готово",
            "готово: view/click/favorite/to_cart",
            "готово с ограничением: widget_name сохраняется как диагностический контекст",
            "готово: search хранить отдельно для будущего co-search",
            "условно: правила описаны, но итоговую null-count таблицу по critical columns стоит сохранить",
            "требует preprocessing decision: p99+ user cap/flag",
            "требует preprocessing decision: min_unique_users и normalization обязательны",
            "blocker для финальной оценки: найден data/path inconsistency; нужен sync данных или фильтр до matched_items",
            "готово с ограничением: текущая проверка day-bounded",
            "условно: после фиксации user/session/item thresholds и catalog invariant",
            "условно: после preprocessing decisions; не финальный baseline/evaluation до решения catalog mismatch",
        ],
    }
)


Readiness checklist показывает, что уже готово, а какие условия нужно закрыть до baseline/evaluation.


In [ ]:
baseline_readiness


Чеклист показывает, что EDA готов, но переход к финальному baseline/evaluation остаётся условным. Главные условия: решить catalog-behavior mismatch, зафиксировать thresholds для пользователей/сессий/товаров и не трактовать day-bounded session check как полную production sessionization.


## EDA conclusions

EDA в целом пригоден для перехода к следующему этапу, но есть существенная проблема: количество уникальных товаров в справочнике и логах поведения не совпадают. В логах есть 161,218 уникальных `item_id`, в справочнике - 130,035, пересечение составляет только 72,793 товара (45,15% уникальных товаров от товаров из логов и 55,98 от товаров из справочника), 88,425 есть в логах, но отсутствует в справочнике, 57,242 есть в справочнике, но нет в логах. Поэтому необходимо синхронизировать данные или учитывать товары из пересечения.

### 1. Product information

- Справочник содержит 130,035 строк и 130,035 уникальных `item_id`.
- Колонки соответствуют ожидаемым: `item_id`, `name`, `brand`, `type`, `category_id`, `category_name`.
- значения null есть только у бренда - 227 и у названия категории - 89. Но для бренда нужно учитывать не только `null`, но и строковое значение `Нет бренда`, которое видно в топе брендов как отдельная крупная группа. По смыслу это отсутствующий бренд, поэтому для fallback/business rules и интерпретации рекомендаций их нужно привести к единому значению.
- `category_id` и `type` можно использовать как стабильные признаки для fallback/business rules; `brand` пригоден только после такой смысловой нормализации.

### 2. User actions

- Лог содержит 561,836,992 строки, 305 parquet-файлов и покрывает 61 день за период 2024-03-01 - 2024-04-30.
- типы действий ожидаемые: `view`, `search`, `to_cart`, `click`, `favorite`.
- Для первого direct item-to-item graph используются только действия `view`, `click`, `favorite`, `to_cart`, так как search не связан с конкретным товаром.
- После исключения search и строк без item_id остаётся 529,562,798 товарных событий у 2,948,770 пользователей.
- `search` не должен входить в первый direct co-visitation graph; его корректнее хранить отдельно для будущего co-search/query-intent сигнала.
- В логах есть 161,218 уникальных `item_id`

### 3. Widget context

- почти все прямые действия с товарами пришли из `search_catalog_listing`, то есть происходят в каталожной/поисковой выдаче: 526,452,168 событий (99.41%).
- `product_detailed_page` или иначе действия, происходящие на карточке товара, дают 3,110,630 событий (0.59%).
- На MVP-этапе не вводим отдельный score по виджетам, но `widget_name` нужно сохранить для диагностики и offline evaluation: смысл `click/view/to_cart` может отличаться в разных местах интерфейса.

### 4. Missing values and search quality

- `search` без `item_id` допустим, это не прямое товарное взаимодействие.
- Товарные события без `item_id` нужно исключать из direct item graph.
- Критичные пропуски user_id, timestamp, action_type равны 0.
- Search quality приемлемый: `search_events` = 32,274,194, `search_users` = 2,623,053, `unique_queries_normalized` = 2,761,580.
- Невалидных поисковых запросов мало: `NULL` rows = 84,215, blank/empty after strip = 773, всего invalid = 84,988, доля = 0.26%.

### 5. Users and anomalous activity

- Распределение пользовательской активности сильно скошено.
- `events_p50` = 32, `events_p90` = 418, `events_p95` = 819, `events_p99` = 2,251 то есть половина пользователей сделала не больше 32 событий, 90%-не больше 418, 95%-не больше 819 и 99% не больше 2251
- p99+ сегмент: 29,494 пользователя генерируют 125,272,315 событий, то есть 23.66%.
- Самый активный пользователь имеет 20,309,379 событий (3.84 от всех), взаимодействовал с 67,820 уникальных товаров. Это подтверждает bot-like или technical traffic risk.
- До генерации item pairs нужно ввести p99+ user flag/cap (установить флаг или ограничить их вклад), иначе один экстремальный пользователь может создать непропорционально много шумных co-visitation пар.

### 6. Items, catalog reconciliation and popularity bias

- Popularity bias подтверждён: среди самых популярных товаров есть такие, которые встретились в сотнях тысяч пользовательских действий, и с ними взаимодействовали более 100 тысяч разных пользователей.
- Raw `pair_count` нельзя использовать как финальный score без нормализации; для первого baseline нужны `min_pair_count`, обязательный `min_unique_users` и cosine-like popularity normalization. Это нужно, чтобы не учитывать пары которые встретились слишком много раз, учитывать те которые встретились у достаточного числа разных пользователей и сделать нормализацию снижающую преимущество просто популярных товаров.
- Вывыявлена существенную несовместимость каталога и поведения пользователей:
  - `catalog_items` = 130,035;
  - `behavior_items` = 161,218;
  - `matched_items` = 72,793-совпадающие
  - `behavior_items_without_product` = 88,425 - есть в логах, но нет в справочнике
  - `catalog_items_without_behavior` = 57,242 - есть в справочнике, но нет в логах
- Это блокирует интерпретацию coverage, fallback и финальную оценку рекомендаций без явного решения: синхронизация данных или работа только на пересечении item_id.
- Fallback нужен как обязательный слой для coverage, top-k completion и cold-start, но его влияние на качество нужно измерять отдельно после построения первого baseline.

### 7. Sessions

- в качестве стартового значения для первого co-visitation baseline выбираем `session_timeout_minutes = 30`, Это значение не считается финально оптимальным: после построения baseline timeout нужно подобрать по validation-метрике. Для проверки стоит взять несколько кандидатов: `5`, `10`, `15`, `30`, `45` и `60` минут.
- Session feasibility подтверждает пригодность `timestamp` для построения сессий: события можно сортировать по времени, а разрывы по времени между действиями считаются. Но проверка диагностическая и делается внутри одного дня, не перенося сессии через границы суток.
- Production preprocessing должен явно переносить сессии через полночь.
- Для контроля комбинаторного взрыва item pairs `max_items_per_session` нужно выбирать по распределению `unique_items_per_session` на user-sample. Это EDA-диагностика для стартового cap, а не финальный оптимум качества.


### 8. Preprocessing decisions

По итогам EDA перед построением baseline нужно зафиксировать следующие preprocessing-решения.

**Удаляем:**

- `search`-события из первого direct item-to-item graph, потому что у них нет конкретного `item_id`.
- Direct item events (`view`, `click`, `favorite`, `to_cart`) без `item_id`, потому что по ним невозможно построить товарные пары.
- Пустые или невалидные `search_query` для уже будущих search/query-intent признаков.

**Оставляем отдельно:**

- `search`-события не для первого item-to-item graph, а для будущего query-intent/co-search сигнала.
- `widget_name` - для диагностики, offline evaluation и возможных widget-specific разрезов в следующих версиях.
- p99+ users - не обязательно удаляем полностью, но помечаем флагом или ограничиваем их вклад через cap.
- Товары без совпадения в каталоге - только после отдельного решения: либо исключать из baseline/evaluation, либо обрабатывать как товары без каталожных признаков.

**Нормализуем:**

- `brand`: `null` и `Нет бренда` считаем одним missing-состоянием.
- `search_query`: приводим к нормализованному виду - trim, lower-case, удаление пустых строк.
- Пользовательские сессии: сортируем события по `user_id` и `timestamp`, режем по выбранному timeout, а максимальную длину сессии ограничиваем cap, выбранным по user-sampled `unique_items_per_session` p95/p99.

**Требует отдельного решения перед baseline/evaluation:**

- нужно выбрать стратегию для товаров, которые есть в `user_actions`, но отсутствуют в `product_information`: синхронизировать источники данных, ограничиться `matched_items` или отдельно учитывать товары без каталожных признаков.
- Session timeout после построения baseline timeout нужно подобрать по validation-метрике. Для проверки стоит взять несколько кандидатов: `5`, `10`, `15`, `30`, `45` и `60` минут.

### 9. Baseline decisions


**Первый baseline:** item-to-item co-visitation на прямых товарных действиях пользователей: `view`, `click`, `favorite`, `to_cart`. `search` не включается в первый direct item graph, потому что это query-intent сигнал, а не прямое взаимодействие с товаром. Его нужно сохранить отдельно для будущего co-search/query-intent baseline.

В первом baseline игнорируем несогласованность товаров в логах поведения и в справочнике товаров.

**Score:** raw `pair_count` не используем как финальный score, потому что он усиливает самые популярные товары и превращает similarity почти в popularity ranking. Для первого baseline нужно использовать score без весов событий, но с нормализацией по популярности.

**Ограничения:**

- catalog-behavior mismatch - главный blocker для финальной evaluation; нужно синхронизировать данные или явно ограничиться `matched_items`;
- fallback-rate пока не измерен; после построения co-visitation graph нужно посчитать `share_items_with_less_than_top_k`, `empty_recommendations_share`, `avg_filled_k`;
- popularity bias подтверждён, поэтому raw `pair_count` использовать нельзя;
- p99+ users искажают пары, поэтому до генерации item pairs нужен cap/flag;
- sessionization пока проверена day-bounded, production preprocessing должен строить полный timeline пользователя и учитывать переход через полночь;
- `widget_name` сохраняем для диагностики и будущей evaluation.
